In [1]:
# Imports y configuración
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Librerías cargadas")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

C:\Users\diego.perezp\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\diego.perezp\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Librerías cargadas
Pandas: 3.0.2
NumPy: 1.26.4


In [2]:
# Ruta del archivo
archivo_emat = r"C:\Users\diego.perezp\Documents\BBDD_tesis_doctorado_german\consolidado_ReporteDinamicoEMAT_2017_2026.xlsx"

# Verificar existencia
if Path(archivo_emat).exists():
    print(f"Archivo encontrado: {Path(archivo_emat).name}")
    print(f"Tamaño: {Path(archivo_emat).stat().st_size / 1024:.2f} KB")
else:
    print(f"ERROR: Archivo no encontrado en {archivo_emat}")

Archivo encontrado: consolidado_ReporteDinamicoEMAT_2017_2026.xlsx
Tamaño: 14.43 KB


In [3]:
# Lectura del archivo
df_raw = pd.read_excel(archivo_emat, sheet_name='Hoja1')

print(f"Dimensiones originales: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print("\nPrimeras 3 filas:")
display(df_raw.head(3))

Dimensiones originales: 21 filas × 45 columnas

Primeras 3 filas:


,Unnamed: 0,N.,AGRUPACIÓN,LLEGADAS,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,PERNOCTACIONES,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,ESTANCIA MEDIA,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,TASA DE OCUPACIÓN EN HABITACIONES,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,TASA DE OCUPACIÓN EN PLAZAS,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,REVPAR,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,ADR,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44
0,AÑO,NaN,NaN,Chi.,Var. 12 meses (%),Ext.,Var. 12 meses (%),Total,Var. 12 meses (%),Chi.,Var. 12 meses (%),Ext.,Var. 12 meses (%),Total,Var. 12 meses (%),Chi.,Var. 12 meses (%),Ext.,Var. 12 meses (%),Total,Var. 12 meses (%),Hotel,Var. 12 meses (pp),Otros,Var. 12 meses (pp),Total,Var. 12 meses (pp),Hotel,Var. 12 meses (pp),Otros,Var. 12 meses (pp),Total,Var. 12 meses (pp),Hotel,Var. 12 meses (%),Otros,Var. 12 meses (%),Total,Var. 12 meses (%),Hotel,Var. 12 meses (%),Otros,Var. 12 meses (%),Total,Var. 12 meses (%)
1,2017,--,NaN,510599,NaN,177722,NaN,688321,NaN,1153322,NaN,334570,NaN,1487892,NaN,2.30,NaN,1.90,NaN,2.20,NaN,41.10,NaN,31.80,NaN,36.10,NaN,28.20,NaN,21,NaN,23.50,NaN,26375,NaN,13276,NaN,19372,NaN,64214,NaN,41783,NaN,53661,NaN
2,2017,1,La Araucanía,510599,NaN,177722,NaN,688321,NaN,1153322,NaN,334570,NaN,1487892,NaN,2.26,NaN,1.88,NaN,2.16,NaN,41.07,NaN,31.80,NaN,36.10,NaN,28.20,NaN,21,NaN,23.50,NaN,26375,NaN,13276,NaN,19372,NaN,64214,NaN,41783,NaN,53661,NaN


In [4]:
# Extraer datos (excluir fila 0 que es encabezado)
df_datos = df_raw.iloc[1:].copy()
df_datos.reset_index(drop=True, inplace=True)

# Mantener solo filas con N. = 1 (La Araucanía)
df_clean = df_datos[df_datos.iloc[:, 1] == 1].copy()
df_clean.reset_index(drop=True, inplace=True)

print(f"Filas después de filtrar: {len(df_clean)}")

Filas después de filtrar: 10


In [5]:
# Construir nombres de columnas
fila_subencabezados = df_raw.iloc[0].tolist()
nombres_principales = df_raw.columns.tolist()

nombres_finales = []
principal_actual = None

for i, (principal, subencabezado) in enumerate(zip(nombres_principales, fila_subencabezados)):
    if i < 3:
        if i == 0:
            nombres_finales.append('AÑO')
        elif i == 1:
            nombres_finales.append('FRECUENCIA')
        elif i == 2:
            nombres_finales.append('AGRUPACION')
    else:
        if 'Unnamed' not in str(principal):
            principal_actual = str(principal).strip().upper()
            principal_actual = principal_actual.replace(' ', '_').replace('Ó', 'O').replace('Í', 'I')
        
        sub_limpio = str(subencabezado).strip()
        
        mapeo_sufijos = {
            'Chi.': 'Chileno',
            'Ext.': 'Extranjero',
            'Total': 'Total',
            'Var. 12 meses (%)': 'Var_12m_pct',
            'Var. 12 meses (pp)': 'Var_12m_pp',
            'Hotel': 'Hotel',
            'Otros': 'Otros'
        }
        
        sufijo = mapeo_sufijos.get(sub_limpio, sub_limpio)
        
        if sufijo == 'Var_12m_pct' or sufijo == 'Var_12m_pp':
            if len(nombres_finales) > 0:
                nombre_anterior = nombres_finales[-1]
                nombre_final = f"{nombre_anterior}_Var"
            else:
                nombre_final = f"{principal_actual}_{sufijo}"
        else:
            nombre_final = f"{principal_actual}_{sufijo}"
        
        nombres_finales.append(nombre_final)

# Asignar nombres
df_clean.columns = nombres_finales

print(f"Columnas generadas: {len(nombres_finales)}")
print(f"Primeras 10: {nombres_finales[:10]}")

Columnas generadas: 45
Primeras 10: ['AÑO', 'FRECUENCIA', 'AGRUPACION', 'LLEGADAS_Chileno', 'LLEGADAS_Chileno_Var', 'LLEGADAS_Extranjero', 'LLEGADAS_Extranjero_Var', 'LLEGADAS_Total', 'LLEGADAS_Total_Var', 'PERNOCTACIONES_Chileno']


In [6]:
# Conversión de tipos
df_clean['AÑO'] = pd.to_numeric(df_clean['AÑO'], errors='coerce').astype('Int64')
df_clean['FRECUENCIA'] = df_clean['FRECUENCIA'].astype(str)
df_clean['AGRUPACION'] = df_clean['AGRUPACION'].astype(str)

# Convertir columnas numéricas
for col in df_clean.columns[3:]:
    try:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    except Exception as e:
        print(f"Error en {col}: {e}")

print("Conversión de tipos completada")
print(f"\nDimensiones wide: {df_clean.shape[0]} filas × {df_clean.shape[1]} columnas")
display(df_clean.head())

Conversión de tipos completada

Dimensiones wide: 10 filas × 45 columnas


,AÑO,FRECUENCIA,AGRUPACION,LLEGADAS_Chileno,LLEGADAS_Chileno_Var,LLEGADAS_Extranjero,LLEGADAS_Extranjero_Var,LLEGADAS_Total,LLEGADAS_Total_Var,PERNOCTACIONES_Chileno,PERNOCTACIONES_Chileno_Var,PERNOCTACIONES_Extranjero,PERNOCTACIONES_Extranjero_Var,PERNOCTACIONES_Total,PERNOCTACIONES_Total_Var,ESTANCIA_MEDIA_Chileno,ESTANCIA_MEDIA_Chileno_Var,ESTANCIA_MEDIA_Extranjero,ESTANCIA_MEDIA_Extranjero_Var,ESTANCIA_MEDIA_Total,ESTANCIA_MEDIA_Total_Var,TASA_DE_OCUPACION_EN_HABITACIONES_Hotel,TASA_DE_OCUPACION_EN_HABITACIONES_Hotel_Var,TASA_DE_OCUPACION_EN_HABITACIONES_Otros,TASA_DE_OCUPACION_EN_HABITACIONES_Otros_Var,TASA_DE_OCUPACION_EN_HABITACIONES_Total,TASA_DE_OCUPACION_EN_HABITACIONES_Total_Var,TASA_DE_OCUPACION_EN_PLAZAS_Hotel,TASA_DE_OCUPACION_EN_PLAZAS_Hotel_Var,TASA_DE_OCUPACION_EN_PLAZAS_Otros,TASA_DE_OCUPACION_EN_PLAZAS_Otros_Var,TASA_DE_OCUPACION_EN_PLAZAS_Total,TASA_DE_OCUPACION_EN_PLAZAS_Total_Var,REVPAR_Hotel,REVPAR_Hotel_Var,REVPAR_Otros,REVPAR_Otros_Var,REVPAR_Total,REVPAR_Total_Var,ADR_Hotel,ADR_Hotel_Var,ADR_Otros,ADR_Otros_Var,ADR_Total,ADR_Total_Var
0,2017,1,La Araucanía,510599.00,NaN,177722.00,NaN,688321,NaN,1153322.00,NaN,334570.00,NaN,1487892,NaN,2.26,NaN,1.88,NaN,2.16,NaN,41.07,NaN,31.80,NaN,36.10,NaN,28.20,NaN,21.00,NaN,23.50,NaN,26375.00,NaN,13276.00,NaN,19372,NaN,64214.00,NaN,41783.00,NaN,53661,NaN
1,2018,1,La Araucanía,522596.00,0.02,104756.00,-0.41,627352,-0.09,1162797.00,0.01,212424.00,-0.37,1375221,-0.08,2.23,-0.01,2.03,0.08,2.19,0.01,39.59,-3.61,27.50,-13.42,33.10,-0.08,26.60,-5.51,19.00,-9.75,21.70,-7.82,25899.00,-0.02,11587.00,-0.13,18257,-0.06,65417.00,0.02,42118.00,0.01,55091,0.03
2,2019,1,La Araucanía,450525.00,-0.14,73543.00,-0.30,524068,-0.16,928156.00,-0.20,142297.00,-0.33,1070453,-0.22,2.06,-0.07,1.93,-0.05,2.04,-0.07,36.16,-8.66,20.60,-25.29,27.80,-0.16,24.40,-8.37,12.70,-32.97,16.80,-22.46,24040.00,-0.07,8982.00,-0.22,15928,-0.13,66475.00,0.02,43706.00,0.04,57392,0.04
3,2020,1,La Araucanía,NaN,NaN,NaN,NaN,226065,-0.57,NaN,NaN,NaN,NaN,501018,-0.53,NaN,NaN,NaN,NaN,2.22,0.09,NaN,NaN,NaN,NaN,28.50,0.03,NaN,NaN,NaN,NaN,19.70,17.12,NaN,NaN,NaN,NaN,15509,-0.03,NaN,NaN,NaN,NaN,54443,-0.05
4,2021,1,La Araucanía,NaN,NaN,NaN,NaN,261326,0.16,NaN,NaN,NaN,NaN,553851,0.11,NaN,NaN,NaN,NaN,2.12,-0.04,NaN,NaN,NaN,NaN,28.90,0.01,NaN,NaN,NaN,NaN,16.90,-14.22,NaN,NaN,NaN,NaN,14743,-0.05,NaN,NaN,NaN,NaN,51029,-0.06


In [7]:
# TRANSFORMACIÓN A LONG FORMAT (MELT)
# Columnas que mantener como identificadores
id_vars = ['AÑO', 'FRECUENCIA', 'AGRUPACION']

# Todas las demás columnas se convertirán en filas
value_vars = [col for col in df_clean.columns if col not in id_vars]

# Aplicar melt
df_long = df_clean.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='VARIABLE',
    value_name='VALOR'
)

# Ordenar por año y variable
df_long = df_long.sort_values(['AÑO', 'VARIABLE']).reset_index(drop=True)

print("TRANSFORMACIÓN A LONG FORMAT COMPLETADA")
print(f"\nDimensiones long: {df_long.shape[0]} filas × {df_long.shape[1]} columnas")
print(f"\nEstructura:")
print(f"- {len(df_long['AÑO'].unique())} años únicos")
print(f"- {len(df_long['VARIABLE'].unique())} variables únicas")
print(f"- {df_long.shape[0]} observaciones totales")

print("\nPrimeras 20 filas:")
display(df_long.head(20))

TRANSFORMACIÓN A LONG FORMAT COMPLETADA

Dimensiones long: 420 filas × 5 columnas

Estructura:
- 10 años únicos
- 42 variables únicas
- 420 observaciones totales

Primeras 20 filas:


,AÑO,FRECUENCIA,AGRUPACION,VARIABLE,VALOR
0,2017,1,La Araucanía,ADR_Hotel,64214.00
1,2017,1,La Araucanía,ADR_Hotel_Var,NaN
2,2017,1,La Araucanía,ADR_Otros,41783.00
3,2017,1,La Araucanía,ADR_Otros_Var,NaN
4,2017,1,La Araucanía,ADR_Total,53661.00
5,2017,1,La Araucanía,ADR_Total_Var,NaN
6,2017,1,La Araucanía,ESTANCIA_MEDIA_Chileno,2.26
7,2017,1,La Araucanía,ESTANCIA_MEDIA_Chileno_Var,NaN
8,2017,1,La Araucanía,ESTANCIA_MEDIA_Extranjero,1.88
9,2017,1,La Araucanía,ESTANCIA_MEDIA_Extranjero_Var,NaN


In [8]:
# Verificar estructura
print("VERIFICACIÓN DE ESTRUCTURA LONG")
print("\nAños disponibles:")
print(df_long['AÑO'].unique())

print("\nEjemplo: LLEGADAS_Total por año")
ejemplo = df_long[df_long['VARIABLE'] == 'LLEGADAS_Total'][['AÑO', 'VARIABLE', 'VALOR']]
display(ejemplo)

print("\nValores nulos por columna:")
print(df_long.isnull().sum())

VERIFICACIÓN DE ESTRUCTURA LONG

Años disponibles:
<IntegerArray>
[2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Length: 10, dtype: Int64

Ejemplo: LLEGADAS_Total por año


,AÑO,VARIABLE,VALOR
16,2017,LLEGADAS_Total,688321.00
58,2018,LLEGADAS_Total,627352.00
100,2019,LLEGADAS_Total,524068.00
142,2020,LLEGADAS_Total,226065.00
184,2021,LLEGADAS_Total,261326.00
226,2022,LLEGADAS_Total,372511.00
268,2023,LLEGADAS_Total,400075.00
310,2024,LLEGADAS_Total,491448.00
352,2025,LLEGADAS_Total,570271.00
394,2026,LLEGADAS_Total,175127.00



Valores nulos por columna:
AÑO             0
FRECUENCIA      0
AGRUPACION      0
VARIABLE        0
VALOR         217
dtype: int64


In [9]:
# Guardar a CSV (long format)
directorio_salida = Path(archivo_emat).parent
archivo_salida = directorio_salida / 'emat_transformado_long_2017_2026.csv'

df_long.to_csv(archivo_salida, index=False, encoding='utf-8')

print("ARCHIVO GUARDADO")
print(f"Ubicación: {archivo_salida}")
print(f"Filas: {df_long.shape[0]}")
print(f"Columnas: {df_long.shape[1]}")
print(f"Tamaño: {archivo_salida.stat().st_size / 1024:.2f} KB")

ARCHIVO GUARDADO
Ubicación: C:\Users\diego.perezp\Documents\BBDD_tesis_doctorado_german\emat_transformado_long_2017_2026.csv
Filas: 420
Columnas: 5
Tamaño: 20.94 KB


In [ ]:
# OPCIONAL: Filtrar solo variables sin variaciones para análisis inicial
df_long_sin_var = df_long[~df_long['VARIABLE'].str.contains('_Var', na=False)].copy()

print("DataFrame sin variables de variación:")
print(f"Filas: {df_long_sin_var.shape[0]}")
print(f"Variables únicas: {len(df_long_sin_var['VARIABLE'].unique())}")

print("\nPrimeras 20 filas:")
display(df_long_sin_var.head(20))

In [ ]:
# Variables disponibles
print("VARIABLES DISPONIBLES EN EL DATASET:")
print("\nTotal:", len(df_long['VARIABLE'].unique()))
print("\nLista completa:")
for i, var in enumerate(sorted(df_long['VARIABLE'].unique()), 1):
    print(f"{i:2d}. {var}")